# Dev experiments: OSTrack + V-JEPA (Kaggle, self-contained)

Everything in one session, no notebook/dataset inputs (mounting large outputs fails on Kaggle):

| step | what | time |
|---|---|---|
| 0 | self-test on 30 frames (fails within minutes if something is broken) | ~5 min |
| 1 | V-JEPA cache for the 40 dev videos | ~1 h |
| 2 | OSTrack + V-JEPA re-localization on dev (no training) | ~30 min |
| 3 | training samples from 280 train videos | ~1 h |
| 4 | train fusion_v1 | ~30-60 min |
| 5 | OSTrack + fusion_v1 on dev | ~30 min |

Results are packed into `/kaggle/working/dev_experiments.zip` after **every** step, so a failure later keeps earlier results.

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, no inputs needed. Then **Save Version -> Save & Run All**.

In [ ]:
# fail fast with a clear message if the session has no internet
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On '
                     '(needs a phone-verified Kaggle account), then run again.')

In [ ]:
import subprocess  # check=True stops the run here if cloning fails
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown "transformers>=4.53"
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, glob, subprocess, torch, numpy as np
os.chdir('/kaggle/working/Object-Tracking')
os.makedirs('/kaggle/tmp', exist_ok=True)
W = '/kaggle/working'
CKPT = f'{W}/OSTrack_ep0300.pth.tar'
CACHE = '/kaggle/tmp/cache/vjepa'
DEV = 'configs/splits/dev_occlusion.txt'
N_GPU = max(torch.cuda.device_count(), 1)

def run_shards(name, args):
    """Run a script once per GPU (each takes every N-th video), wait, show the log tail."""
    procs = []
    for i in range(N_GPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'{W}/logs/{name}_shard{i}.log', 'w')
        procs.append(subprocess.Popen(['python'] + args + ['--shard', str(i), '--num-shards', str(N_GPU)],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    os.system(f"grep -hE 'FAILED|Error|Traceback' {W}/logs/{name}_shard*.log | head -n 20")
    os.system(f"tail -n 2 {W}/logs/{name}_shard*.log")
    print(name, 'exit codes', codes)
    return codes

def pack():
    """Zip everything worth keeping (small files only) - called after every step."""
    os.system(f'cd {W} && rm -f dev_experiments.zip && zip -qr dev_experiments.zip results_raw logs output 2>/dev/null; ls -la {W}/dev_experiments.zip')

os.makedirs(f'{W}/logs', exist_ok=True)

## 0. Self-test (30 frames): plain OSTrack, re-localization and zero-initialised fusion must all run; fusion before training must equal plain OSTrack

In [ ]:
import sys; sys.path.insert(0, '.')
from src.tracker.fusion import FusionV1
os.makedirs('/kaggle/tmp/selftest/cache', exist_ok=True)
np.savez('/kaggle/tmp/selftest/cache/airplane-1.npz', frames=np.arange(0, 3000, 8),
         maps=np.random.randn(375, 16, 16, 1024).astype(np.float16))
torch.save(FusionV1().state_dict(), '/kaggle/tmp/selftest/fusion_zero.pth')
base = ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', 'airplane-1', '--max-frames', '30',
        '--work', '/kaggle/tmp/selftest/work']
for name, extra in [('plain', []), ('reloc', ['--vjepa-cache', '/kaggle/tmp/selftest/cache']),
                    ('fusion0', ['--vjepa-cache', '/kaggle/tmp/selftest/cache', '--fusion', '/kaggle/tmp/selftest/fusion_zero.pth'])]:
    subprocess.run(['python'] + base + ['--out', f'/kaggle/tmp/selftest/{name}'] + extra, check=True)
plain = np.loadtxt('/kaggle/tmp/selftest/plain/airplane-1.txt', delimiter='	')
fusion0 = np.loadtxt('/kaggle/tmp/selftest/fusion0/airplane-1.txt', delimiter='	')
diff = np.abs(plain - fusion0).max()
print('max box difference, zero-init fusion vs plain OSTrack:', diff)
assert diff < 1.0, 'zero-initialised fusion should reproduce OSTrack'
print('SELF-TEST OK')

## 1. V-JEPA cache for the dev videos

In [ ]:
for attempt in range(2):  # a re-run only does the videos that failed to download
    run_shards('vjepa_cache', ['scripts/cache_vjepa.py', '--sequences', DEV, '--out', CACHE, '--work', '/kaggle/tmp'])
    n = len(glob.glob(f'{CACHE}/*.npz'))
    print(n, 'of 40 dev videos cached')
    if n == 40:
        break
assert n == 40, 'V-JEPA cache incomplete'

## 2. OSTrack + V-JEPA re-localization on dev (no training)

In [ ]:
run_shards('reloc', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV, '--vjepa-cache', CACHE,
                     '--out', f'{W}/results_raw/ostrack_vjepa_reloc/lasot', '--work', '/kaggle/tmp'])
pack()

## 3. Training samples (280 train videos, 4 per class)

In [ ]:
run_shards('prepare', ['scripts/prepare_fusion_data.py', '--out', '/kaggle/tmp/fusion_data', '--work', '/kaggle/tmp'])
print(len(glob.glob('/kaggle/tmp/fusion_data/motion/*.npz')), 'videos with samples')

## 4. Train fusion_v1 (one GPU)

In [ ]:
subprocess.run(['python', 'scripts/train_fusion.py', '--data', '/kaggle/tmp/fusion_data/motion', '--checkpoint', CKPT,
                '--out', f'{W}/output/fusion_v1'], check=True)
pack()

## 5. OSTrack + fusion_v1 on dev

In [ ]:
run_shards('fusion', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV, '--vjepa-cache', CACHE,
                      '--fusion', f'{W}/output/fusion_v1/fusion.pth',
                      '--out', f'{W}/results_raw/ostrack_fusion_v1/lasot', '--work', '/kaggle/tmp'])
pack()
os.system(f'rm -rf {W}/Object-Tracking {CKPT}; du -sh {W}/*')